# SafeSLM Colab
Run all cells with a T4 GPU runtime. Set `REPO_URL` to your repository.

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
REPO_URL = 'https://github.com/YOUR_USER/YOUR_REPO.git'  # replace once
%cd /content
!test -d SafeSLM || git clone $REPO_URL SafeSLM
%cd /content/SafeSLM
!pip -q install -r requirements.txt

In [ ]:
!PYTHONPATH=src python scripts/smoke_test.py --offline
from safeslm.config import load_config
print(load_config().model_name)

In [ ]:
# Downloads and loads the base model on Colab only
!python scripts/download_model.py
!python scripts/baseline_eval.py

In [ ]:
import json
train=[json.loads(x) for x in open('data/safety_train.jsonl')]
print('train examples:', len(train)); print(json.dumps(train[:2], indent=2))
!python scripts/train.py

In [ ]:
!python scripts/evaluate.py --adapter outputs/safeslm-lora
!python scripts/compare.py --adapter outputs/safeslm-lora

In [ ]:
from IPython.display import Image, display
display(Image('results/comparison.png'))
base=json.load(open('results/base_results.json')); safe=json.load(open('results/safeslm_results.json'))
print('BASE', base['metrics']); print('SAFESLM', safe['metrics'])
for b,s in zip(base['records'][:4], safe['records'][:4]):
    print('\nPROMPT:', b['prompt'], '\nBASE:', b['response'], '\nSAFESLM:', s['response'])

In [ ]:
# Optional: persist artifacts to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs results /content/drive/MyDrive/SafeSLM-results